In [1]:
import os
import cv2
import json
import numpy as np 
import matplotlib.pyplot as plt
import glob
from osgeo import gdal

In [2]:
# Extract X and Y coordinates if available and update dictionary
def add_to_dict(data, itr, key, count):
    try:
        x_points = data[itr]["regions"][count]["shape_attributes"]["all_points_x"]
        y_points = data[itr]["regions"][count]["shape_attributes"]["all_points_y"]
    except:
        print("No BB. Skipping", key)
        return
    
    all_points = []
    for i, x in enumerate(x_points):
        all_points.append([x, y_points[i]])
    
    file_bbs[key] = all_points

In [5]:
#####################################################################
# Convert json to mask
#####################################################################

base_dir = '/data/rrs/seaice/aux_data/s1/tundra_lakes'
region = 'Siberia'

json_path = f'{base_dir}/{region}/json'
img_folder = f'{base_dir}/{region}/tiff_rgb'
masks_folder = f'{base_dir}/{region}/masks_tiff'
os.makedirs(masks_folder, exist_ok=True)

img_format = 'tiff'

json_files = glob.glob(f'{json_path}/*.json')

total_objects = 0

for json_path in json_files:
    count = 0                                           # Count of total images saved
    file_bbs = {}                                       # Dictionary containing polygon coordinates for mask

    # Read JSON file
    with open(json_path) as f:
        data = json.load(f)

    for itr in [list(data.keys())[0]]:
        #print(itr)
        img_fname = itr
        tiff_img_fname = f'{img_folder}/{img_fname}'
        
        img = gdal.Open(tiff_img_fname).ReadAsArray()
        print(f'\nImage filename: {img_fname}')
        
        if len(img.shape)>2:
            MASK_WIDTH, MASK_HEIGHT = img.shape[1], img.shape[2]
        else:
            MASK_WIDTH, MASK_HEIGHT = img.shape
        print(f'WxH: {MASK_WIDTH}x{MASK_HEIGHT}')

        file_name_json = data[itr]["filename"]
        sub_count = 0               # Contains count of masks for a single ground truth image

        if len(data[itr]["regions"]) > 1:
            for _ in range(len(data[itr]["regions"])):
                key = file_name_json[:-4] + "*" + str(sub_count+1)
                add_to_dict(data, itr, key, sub_count)
                sub_count += 1
        else:
            add_to_dict(data, itr, file_name_json[:-4], 0)
    
    total_objects += len(file_bbs)
    print("\nNumber of objects: \n", len(file_bbs))

    #num_masks = itr.split("*")
    mask = np.zeros((MASK_WIDTH, MASK_HEIGHT))

    for itr in file_bbs:
        try:
            arr = np.array(file_bbs[itr])
        except:
            print("Not found:", itr)
            continue
        count += 1
        cv2.fillPoly(mask, [arr], color=(255))

    cv2.imwrite(f'{masks_folder}/{os.path.basename(tiff_img_fname)}', mask)
    
print(f'\nTotal number of objects: {total_objects}')


Image filename: LE07_L1TP_163011_20010714_20200917_02_T1_rgb.tiff
WxH: 8401x8811

Number of objects: 
 118

Image filename: LE07_L1TP_166010_19990916_20200918_02_T1_rgb.tiff
WxH: 8291x8731

Number of objects: 
 107

Image filename: LE07_L1TP_165011_19990808_20201008_02_T1_rgb.tiff
WxH: 8221x8671

Number of objects: 
 116

Image filename: LE07_L1TP_166010_20020706_20200916_02_T1_rgb.tiff
WxH: 8341x8751

Number of objects: 
 117

Total number of objects: 458
